In [14]:
import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer
from pandas_ml import ConfusionMatrix

In [15]:
train = pd.read_csv('UNSW_NB15_training-set.csv')
test = pd.read_csv('UNSW_NB15_testing-set.csv')
combined_data = pd.concat([train, test]).drop(['id'],axis=1)

In [16]:
train.head(3)

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0


In [17]:
# Contaminsation mean pollution (outliers) in data
tmp = train.where(train['attack_cat'] == "Normal").dropna()
contamination = round(1 - len(tmp)/len(train), 2)
print("train contamination ", contamination)

tmp = test.where(test['attack_cat'] == "Normal").dropna()
print("test  contamination ", round(1 - len(tmp)/len(test),2),'\n')

if contamination > 0.5:
    print(f'contamination is {contamination}, which is greater than 0.5. Fixing...')
    contamination = round(1-contamination,2)
    print(f'contamination is now {contamination}')

train contamination  0.55
test  contamination  0.68 

contamination is 0.55, which is greater than 0.5. Fixing...
contamination is now 0.45


In [18]:
from sklearn.preprocessing import LabelEncoder,normalize
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data['attack_cat']

print("attack cat:", set(list(vector))) # use print to make it print on single line 

combined_data['attack_cat'] = le1.fit_transform(vector)
combined_data['proto'] = le.fit_transform(combined_data['proto'])
combined_data['service'] = le.fit_transform(combined_data['service'])
combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data['attack_cat']
print('\nDescribing attack_type: ')
print("min", vector.min())
print("max", vector.max())
print("mode",vector.mode(), "Which is,", le1.inverse_transform(vector.mode()))
print("mode", len(np.where(vector.values==6)[0])/len(vector),"%")

attack cat: {'Shellcode', 'Analysis', 'Fuzzers', 'DoS', 'Backdoor', 'Generic', 'Worms', 'Normal', 'Exploits', 'Reconnaissance'}

Describing attack_type: 
min 0
max 9
mode 0    6
dtype: int32 Which is, ['Normal']
mode 0.3609225646458884 %


In [19]:
le1.inverse_transform([0,1,2,3,4,5,6,7,8,9])
combined_data.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,6,0


In [20]:
## OMITTED: For statistical feature removal

lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
# this is stupid. suppose a feature has a 1.0 (spearman or pearson) correlation, OR conditional probability, when not 0.... That a very useful feature  

lowCORR = list(combined_data.corr().abs().sort_values('attack_cat')['attack_cat'].nsmallest(3).index) # .where(lambda x: x < 0.005).dropna()
# This might be stupid. A Deep MLP (feed forward neural net) may see patterns

drop = set( lowCORR + lowSTD)
drop = {'ackdat', 'ct_ftp_cmd', 'djit', 'is_ftp_login', 'is_sm_ips_ports', 'response_body_len', 'sjit', 'synack', 'tcprtt'}
# print(f'Before {combined_data.shape}')
combined_data_reduced=combined_data.drop(drop,axis=1)
# print(f'After {combined_data.shape}')

In [21]:
combined_data_reduced.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,2,1,1,1,2,0,1,2,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,2,1,1,1,2,0,1,2,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,2,1,1,1,3,0,1,3,6,0


In [22]:
def data_normalization(train_df,dest_feature):
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    for name in dest_feature:
        train_df[name]=train_df[[name]].apply(max_min_scaler)
    return train_df

In [23]:
df = combined_data_reduced
print(len(df.columns.values))
print(df.columns.values)
columns = np.delete(df.columns.values, [34])
print("@@@@@@@@@@@@@@@@@")
print(columns)
print("@@@@@@@@@@@@@@@@@")
combined_data_reduced = data_normalization(df, columns)

35
['dur' 'proto' 'service' 'state' 'spkts' 'dpkts' 'sbytes' 'dbytes' 'rate'
 'sttl' 'dttl' 'sload' 'dload' 'sloss' 'dloss' 'sinpkt' 'dinpkt' 'swin'
 'stcpb' 'dtcpb' 'dwin' 'smean' 'dmean' 'trans_depth' 'ct_srv_src'
 'ct_state_ttl' 'ct_dst_ltm' 'ct_src_dport_ltm' 'ct_dst_sport_ltm'
 'ct_dst_src_ltm' 'ct_flw_http_mthd' 'ct_src_ltm' 'ct_srv_dst' 'attack_cat'
 'label']
@@@@@@@@@@@@@@@@@
['dur' 'proto' 'service' 'state' 'spkts' 'dpkts' 'sbytes' 'dbytes' 'rate'
 'sttl' 'dttl' 'sload' 'dload' 'sloss' 'dloss' 'sinpkt' 'dinpkt' 'swin'
 'stcpb' 'dtcpb' 'dwin' 'smean' 'dmean' 'trans_depth' 'ct_srv_src'
 'ct_state_ttl' 'ct_dst_ltm' 'ct_src_dport_ltm' 'ct_dst_sport_ltm'
 'ct_dst_src_ltm' 'ct_flw_http_mthd' 'ct_src_ltm' 'ct_srv_dst' 'attack_cat']
@@@@@@@@@@@@@@@@@


In [24]:
combined_data_reduced.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,attack_cat,label
0,1.833334e-07,0.901515,0.0,0.5,0.000094,0.0,0.000033,0.0,0.090909,0.996078,...,0.333333,0.0,0.0,0.0,0.015625,0.0,0.0,0.016393,0.666667,0
1,1.333334e-07,0.901515,0.0,0.5,0.000094,0.0,0.000121,0.0,0.125000,0.996078,...,0.333333,0.0,0.0,0.0,0.015625,0.0,0.0,0.016393,0.666667,0
2,8.333335e-08,0.901515,0.0,0.5,0.000094,0.0,0.000073,0.0,0.200000,0.996078,...,0.333333,0.0,0.0,0.0,0.031250,0.0,0.0,0.032787,0.666667,0


In [25]:
def gererated_preprocess(generated_data,lbl):
    """为GAN生成的数据加上attack_type"""
    df = generated_data
    columns = lbl[:-1]
    df["label"] = pd.Series([1]*len(df), index=df.index)
    return df
lbl = combined_data_reduced.columns.values
read_generated_data = pd.read_csv('./output3/fake_examples.csv', sep=",", header=None)
#print(read_generated_data.head(3))
generated_data = gererated_preprocess(read_generated_data,lbl)

In [26]:
#generated_data.head(3)

In [27]:
generated_data.columns = lbl
generated_data.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,attack_cat,label
0,0.489475,0.063405,0.896988,0.064806,0.473199,0.054465,0.048163,0.049579,0.043029,0.050078,...,0.048907,0.209894,0.049757,0.049019,0.049184,0.048131,0.074662,0.038862,0.029490,1
1,0.483195,0.067122,0.893597,0.068472,0.473108,0.051534,0.046145,0.047447,0.043051,0.046717,...,0.039353,0.202525,0.042069,0.042500,0.042830,0.042936,0.068584,0.035609,0.027005,1
2,0.534316,0.033091,0.919638,0.510162,0.441910,0.043101,0.040358,0.041017,0.038022,0.041041,...,0.038673,0.205714,0.040769,0.040100,0.039861,0.040056,0.065354,0.032824,0.024251,1


In [28]:
generated_data.shape

(10000, 35)

In [29]:
combined_data_reduced = combined_data_reduced.append(generated_data)

data_x = combined_data_reduced.drop(['attack_cat','label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,['label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

In [30]:
#y_train = pd.to_numeric(y_train['label'].astype(str)) 

In [31]:
print(X_train.columns.values)

print(y_train.columns.values)

['dur' 'proto' 'service' 'state' 'spkts' 'dpkts' 'sbytes' 'dbytes' 'rate'
 'sttl' 'dttl' 'sload' 'dload' 'sloss' 'dloss' 'sinpkt' 'dinpkt' 'swin'
 'stcpb' 'dtcpb' 'dwin' 'smean' 'dmean' 'trans_depth' 'ct_srv_src'
 'ct_state_ttl' 'ct_dst_ltm' 'ct_src_dport_ltm' 'ct_dst_sport_ltm'
 'ct_dst_src_ltm' 'ct_flw_http_mthd' 'ct_src_ltm' 'ct_srv_dst']
['label']


In [32]:
y_train = y_train.values.flatten()
y_test = y_test.values.flatten()

In [33]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape) # test is larger... good 
print(y_test.shape)
# y_train.min()

(214138, 33)
(214138,)
(53535, 33)
(53535,)


In [34]:
y_train.head(3)

AttributeError: 'numpy.ndarray' object has no attribute 'head'

In [35]:
# traindata = pd.read_csv('UNSW_NB15_training-set.csv', header=None)
# testdata = pd.read_csv('UNSW_NB15_testing-set.csv', header=None)
# traindata = pd.read_csv('kddtrain.csv', header=None)
# testdata = pd.read_csv('kddtest.csv', header=None)

# X = traindata.iloc[:,1:42]
# Y = traindata.iloc[:,0]
# C = testdata.iloc[:,0]
# T = testdata.iloc[:,1:42]
X = X_train
Y = y_train
C = y_test
T = X_test

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)


traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)




In [36]:
model = LogisticRegression()
model.fit(traindata, trainlabel)
print(model)

# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)


cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='warn',
          n_jobs=None, penalty='l2', random_state=None, solver='warn',
          tol=0.0001, verbose=0, warm_start=False)
(53535,)
(53535,)
population: 53535
P: 34949
N: 18586
PositiveTest: 39262
NegativeTest: 14273
TP: 34030
TN: 13354
FP: 5232
FN: 919
TPR: 0.973704540902
TNR: 0.718497794039
PPV: 0.866741378432
NPV: 0.935612695299
FPR: 0.281502205961
FDR: 0.133258621568
FNR: 0.0262954590975
ACC: 0.885103203512
F1_score: 0.917114713452
MCC: 0.745245840838
informedness: 0.692202334941
markedness: 0.802354073731
prevalence: 0.652825254506
LRP: 3.45895882974
LRN: 0.0365978285747
DOR: 94.512679152
FOR: 0.0643873047012
Predicted  False   True  __all__
Actual                          
False      13354   5232    18586
True         919  34030    34949
__all__    14273  39262    53535
(53535,)
(53535,)
******************************************************

In [67]:



# fit a Naive Bayes model to the data
model = GaussianNB()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)

expected = expected.flatten()
#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



# fit a k-nearest neighbor model to the data
model = KNeighborsClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()


# cm = metrics.confusion_matrix(expected, predicted)
# print(cm)
# tpr = float(cm[0][0])/np.sum(cm[0])
# fpr = float(cm[1][1])/np.sum(cm[1])
# print("%.3f" %tpr)
# print("%.3f" %fpr)
# print("Accuracy")
# print("%.3f" %ACC)
# print("precision")
# print("%.3f" %precision)
# print("recall")
# print("%.3f" %recall)
# print("f-score")
# print("%.3f" %f1)
# print("fpr")
# print("%.3f" %fpr)
# print("tpr")
# print("%.3f" %tpr)
print("***************************************************************")



model = DecisionTreeClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()


print("***************************************************************")





print("AdaBoostClassifier(n_estimators=100)")
model = AdaBoostClassifier(n_estimators=100)
model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



print("RandomForestClassifier(n_estimators=100)")
model = RandomForestClassifier(n_estimators=100)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")










D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GaussianNB(priors=None, var_smoothing=1e-09)
(53535,)
(53535,)
population: 53535
P: 34949
N: 18586
PositiveTest: 20244
NegativeTest: 33291
TP: 18858
TN: 17200
FP: 1386
FN: 16091
TPR: 0.539586254256
TNR: 0.925427741311
PPV: 0.93153526971
NPV: 0.516656153315
FPR: 0.0745722586893
FDR: 0.0684647302905
FNR: 0.460413745744
ACC: 0.673540674325
F1_score: 0.683347525954
MCC: 0.456525228656
informedness: 0.465013995567
markedness: 0.448191423024
prevalence: 0.652825254506
LRP: 7.23575044849
LRN: 0.497514527814
DOR: 14.5437973043
FOR: 0.483343846685
Predicted  False   True  __all__
Actual                          
False      17200   1386    18586
True       16091  18858    34949
__all__    33291  20244    53535
(53535,)
(53535,)
***************************************************************


D:\Anaconda3\envs\TF_36m\lib\site-packages\ipykernel_launcher.py:36: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().


KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=5, p=2,
           weights='uniform')
(53535,)
(53535,)
population: 53535
P: 34949
N: 18586
PositiveTest: 34938
NegativeTest: 18597
TP: 32838
TN: 16486
FP: 2100
FN: 2111
TPR: 0.939597699505
TNR: 0.887011729259
PPV: 0.939893525674
NPV: 0.886487067807
FPR: 0.112988270741
FDR: 0.0601064743259
FNR: 0.060402300495
ACC: 0.921341178668
F1_score: 0.939745589308
MCC: 0.826495003202
informedness: 0.826609428764
markedness: 0.826380593481
prevalence: 0.652825254506
LRP: 8.3158870681
LRN: 0.0680963943346
DOR: 122.119344928
FOR: 0.113512932193
Predicted  False   True  __all__
Actual                          
False      16486   2100    18586
True        2111  32838    34949
__all__    18597  34938    53535
(53535,)
(53535,)
***************************************************************
DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
           

D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


population: 53535
P: 34949
N: 18586
PositiveTest: 35313
NegativeTest: 18222
TP: 33202
TN: 16475
FP: 2111
FN: 1747
TPR: 0.950012875905
TNR: 0.886419885936
PPV: 0.940220315465
NPV: 0.904126879596
FPR: 0.113580114064
FDR: 0.0597796845354
FNR: 0.0499871240951
ACC: 0.927934995797
F1_score: 0.945091229968
MCC: 0.840380661556
informedness: 0.836432761841
markedness: 0.844347195061
prevalence: 0.652825254506
LRP: 8.36425358198
LRN: 0.0563921510429
DOR: 148.323009981
FOR: 0.0958731204039
Predicted  False   True  __all__
Actual                          
False      16475   2111    18586
True        1747  33202    34949
__all__    18222  35313    53535
(53535,)
(53535,)
***************************************************************
RandomForestClassifier(n_estimators=100)


D:\Anaconda3\envs\TF_36m\lib\site-packages\ipykernel_launcher.py:137: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().


population: 53535
P: 34949
N: 18586
PositiveTest: 34998
NegativeTest: 18537
TP: 33639
TN: 17227
FP: 1359
FN: 1310
TPR: 0.962516810209
TNR: 0.926880447649
PPV: 0.961169209669
NPV: 0.929330528133
FPR: 0.0731195523512
FDR: 0.0388307903309
FNR: 0.0374831897908
ACC: 0.950144765107
F1_score: 0.961842537922
MCC: 0.889948327109
informedness: 0.889397257858
markedness: 0.890499737802
prevalence: 0.652825254506
LRP: 13.1636037046
LRN: 0.0404401558863
DOR: 325.50823349
FOR: 0.0706694718671
Predicted  False   True  __all__
Actual                          
False      17227   1359    18586
True        1310  33639    34949
__all__    18537  34998    53535
(53535,)
(53535,)
***************************************************************


In [68]:
model = svm.SVC(kernel='linear')#调参
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")

D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


population: 53535
P: 34949
N: 18586
PositiveTest: 40442
NegativeTest: 13093
TP: 34888
TN: 13032
FP: 5554
FN: 61
TPR: 0.998254599559
TNR: 0.701172925858
PPV: 0.862667523861
NPV: 0.99534102192
FPR: 0.298827074142
FDR: 0.137332476139
FNR: 0.00174540044064
ACC: 0.895115345101
F1_score: 0.925521613986
MCC: 0.774670764882
informedness: 0.699427525418
markedness: 0.858008545781
prevalence: 0.652825254506
LRP: 3.34057615906
LRN: 0.00248925817908
DOR: 1341.99665874
FOR: 0.00465897807989
Predicted  False   True  __all__
Actual                          
False      13032   5554    18586
True          61  34888    34949
__all__    13093  40442    53535
(53535,)
(53535,)
***************************************************************


In [69]:
def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=33,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

In [70]:
#DNN
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.wrappers.scikit_learn import KerasClassifier
import h5py

from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger

batch_size = 64
checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
#model = KerasClassifier(build_fn=build_model, epochs=2, batch_size=batch_size)
model = KerasClassifier(build_fn=build_model, epochs=10, batch_size=batch_size)
model.fit(traindata, trainlabel,batch_size=batch_size, epochs=10, callbacks=[checkpointer,csv_logger])
#model.save("DNNResult/dnn1layer_model.hdf5")

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
predicted = predicted.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")


Epoch 1/10
214138/214138 [==============================] - 12s 54us/step - loss: 0.2052 - acc: 0.9021

Epoch 00001: loss improved from inf to 0.20519, saving model to ./DNNResult/checkpoint-01.hdf5
Epoch 2/10
214138/214138 [==============================] - 11s 54us/step - loss: 0.1700 - acc: 0.9176

Epoch 00002: loss improved from 0.20519 to 0.17001, saving model to ./DNNResult/checkpoint-02.hdf5
Epoch 3/10
214138/214138 [==============================] - 12s 55us/step - loss: 0.1577 - acc: 0.9248

Epoch 00003: loss improved from 0.17001 to 0.15771, saving model to ./DNNResult/checkpoint-03.hdf5
Epoch 4/10
214138/214138 [==============================] - 12s 55us/step - loss: 0.1503 - acc: 0.9282

Epoch 00004: loss improved from 0.15771 to 0.15028, saving model to ./DNNResult/checkpoint-04.hdf5
Epoch 5/10
214138/214138 [==============================] - 12s 58us/step - loss: 0.1459 - acc: 0.9298

Epoch 00005: loss improved from 0.15028 to 0.14591, saving model to ./DNNResult/checkpoi